In [2]:
!pip3 install omegaconf

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144613 sha256=6055a630a3f708d4191736bea31445d53c2f5dc0486bd962e2ee1a968419d78c
  Stored in directory: c:\users\hp\appdata\local\pip\cache\wheels\1f\be\48\13754633f1d08d1fbfc60d5e80ae1e5d7329500477685286cd
Successfully built antlr4-python3-runtime

   ---------------------------------------- 0/2 [antlr4-python3-runtime]
   ---------------------------------------- 0/2 [antlr4-python3-runtime]
   ---------------------------------------- 0/2 [antlr4-python3-runtime]
   -------------------- ------------------- 1/2 [omegaconf]
   -------------------- ---------------

In [1]:
import importlib
from omegaconf import OmegaConf
import nemo.collections.asr as nemo_asr
model_path = "/data/asr/ams/Conformer-CTC-BPE-v1_95_400.nemo"
asr_model = nemo_asr.models.EncDecCTCModelBPE.restore_from(model_path, map_location='cpu')


asr_model.eval()
asr_model.to('cpu')
asr_model.export('/data/asr/ams/Conformer-CTC-BPE-v1_95_400-averaged.onnx')
#print(OmegaConf.to_yaml(asr_model._cfg))


c:\ProgramData\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


FileNotFoundError: Can't find c:\data\asr\ams\Conformer-CTC-BPE-v1_95_400.nemo

In [ ]:
import os
import onnxruntime
import torch
import numpy as np
import torchaudio
from torchaudio.models.decoder import ctc_decoder
import random
from typing import Optional, Tuple, Union

# Define constants
SAMPLE_RATE = 16000
CHUNK_LEN_SECS = 0.16
OVERLAP_SECS = 1.92
BUFFER_LEN_SECS = CHUNK_LEN_SECS + 2 * OVERLAP_SECS
CHUNK_LEN = int(CHUNK_LEN_SECS * SAMPLE_RATE)
OVERLAP_LEN = int(OVERLAP_SECS * SAMPLE_RATE)
BUFFER_LEN = int(BUFFER_LEN_SECS * SAMPLE_RATE)

class AudioChunkIterator:
    def __init__(self, samples, chunk_len_in_secs, sample_rate):
        self.samples = samples
        self.chunk_len = int(chunk_len_in_secs * sample_rate)
        self.sample_rate = sample_rate
        self.index = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.index + self.chunk_len > len(self.samples[0]):
            raise StopIteration
        chunk = self.samples[:, self.index:self.index + self.chunk_len]
        self.index += self.chunk_len
        return chunk

class FilterbankFeaturesTA(nn.Module):

    def __init__(
        self,
        sample_rate: int = 16000,
        n_window_size: int = 320,
        n_window_stride: int = 160,
        normalize: Optional[str] = "per_feature",
        nfilt: int = 64,
        n_fft: Optional[int] = None,
        preemph: float = 0.97,
        lowfreq: float = 0,
        highfreq: Optional[float] = None,
        log: bool = True,
        log_zero_guard_type: str = "add",
        log_zero_guard_value: Union[float, str] = 2 ** -24,
        dither: float = 1e-5,
        window: str = "hann",
        pad_to: int = 0,
        pad_value: float = 0.0,
        mel_norm="slaney",
        # Seems like no one uses these options anymore. Don't convolute the code by supporting thm.
        use_grads: bool = False,  # Deprecated arguments; kept for config compatibility
        max_duration: float = 16.7,  # Deprecated arguments; kept for config compatibility
        frame_splicing: int = 1,  # Deprecated arguments; kept for config compatibility
        exact_pad: bool = False,  # Deprecated arguments; kept for config compatibility
        nb_augmentation_prob: float = 0.0,  # Deprecated arguments; kept for config compatibility
        nb_max_freq: int = 4000,  # Deprecated arguments; kept for config compatibility
        mag_power: float = 2.0,  # Deprecated arguments; kept for config compatibility
        rng: Optional[random.Random] = None,  # Deprecated arguments; kept for config compatibility
        stft_exact_pad: bool = False,  # Deprecated arguments; kept for config compatibility
        stft_conv: bool = False,  # Deprecated arguments; kept for config compatibility
    ):
        super().__init__()

        # Make sure log zero guard is supported, if given as a string
        supported_log_zero_guard_strings = {"eps", "tiny"}
        if isinstance(log_zero_guard_value, str) and log_zero_guard_value not in supported_log_zero_guard_strings:
            raise ValueError(
                f"Log zero guard value must either be a float or a member of {supported_log_zero_guard_strings}"
            )

        # Copied from `AudioPreprocessor` due to the ad-hoc structuring of the Mel Spec extractor class
        self.torch_windows = {
            'hann': torch.hann_window
        }

        # Ensure we can look up the window function
        if window not in self.torch_windows:
            raise ValueError(f"Got window value '{window}' but expected a member of {self.torch_windows.keys()}")

        self.win_length = n_window_size
        self.hop_length = n_window_stride
        self._sample_rate = sample_rate
        self._normalize_strategy = normalize
        self._use_log = log
        self._preemphasis_value = preemph
        self.log_zero_guard_type = log_zero_guard_type
        self.log_zero_guard_value: Union[str, float] = log_zero_guard_value
        self.dither = dither
        self.pad_to = pad_to
        self.pad_value = pad_value
        self.n_fft = n_fft
        self._mel_spec_extractor: torchaudio.transforms.MelSpectrogram = torchaudio.transforms.MelSpectrogram(
            sample_rate=self._sample_rate,
            win_length=self.win_length,
            hop_length=self.hop_length,
            n_mels=nfilt,
            window_fn=self.torch_windows[window],
            mel_scale="slaney",
            norm=mel_norm,
            n_fft=n_fft,
            f_max=highfreq,
            f_min=lowfreq,
            wkwargs={"periodic": False},
        )

    @property
    def filter_banks(self):
        """ Matches the analogous class """
        return self._mel_spec_extractor.mel_scale.fb

    def _resolve_log_zero_guard_value(self, dtype: torch.dtype) -> float:
        if isinstance(self.log_zero_guard_value, float):
            return self.log_zero_guard_value
        return getattr(torch.finfo(dtype), self.log_zero_guard_value)

    def _apply_dithering(self, signals: torch.Tensor) -> torch.Tensor:
        if self.training and self.dither > 0.0:
            noise = torch.randn_like(signals) * self.dither
            signals = signals + noise
        return signals

    def _apply_preemphasis(self, signals: torch.Tensor) -> torch.Tensor:
        if self._preemphasis_value is not None:
            padded = torch.nn.functional.pad(signals, (1, 0))
            signals = signals - self._preemphasis_value * padded[:, :-1]
        return signals

    def _compute_output_lengths(self, input_lengths: torch.Tensor) -> torch.Tensor:
        out_lengths = input_lengths.div(self.hop_length, rounding_mode="floor").add(1).long()
        return out_lengths

    def _apply_pad_to(self, features: torch.Tensor) -> torch.Tensor:
        # Only apply during training; else need to capture dynamic shape for exported models
        if not self.training or self.pad_to == 0 or features.shape[-1] % self.pad_to == 0:
            return features
        pad_length = self.pad_to - (features.shape[-1] % self.pad_to)
        return torch.nn.functional.pad(features, pad=(0, pad_length), value=self.pad_value)

    def _apply_log(self, features: torch.Tensor) -> torch.Tensor:
        if self._use_log:
            zero_guard = self._resolve_log_zero_guard_value(features.dtype)
            if self.log_zero_guard_type == "add":
                features = features + zero_guard
            elif self.log_zero_guard_type == "clamp":
                features = features.clamp(min=zero_guard)
            else:
                raise ValueError(f"Unsupported log zero guard type: '{self.log_zero_guard_type}'")
            features = features.log()
        return features

    def _extract_spectrograms(self, signals: torch.Tensor) -> torch.Tensor:
        # Complex FFT needs to be done in single precision
        with torch.cuda.amp.autocast(enabled=False):
            features = self._mel_spec_extractor(waveform=signals)
        return features

    def _apply_normalization(self, features: torch.Tensor, lengths: torch.Tensor, eps: float = 1e-5) -> torch.Tensor:
        # For consistency, this function always does a masked fill even if not normalizing.
        mask: torch.Tensor = make_seq_mask_like(lengths=lengths, like=features, time_dim=-1, valid_ones=False)
        features = features.masked_fill(mask, 0.0)
        # Maybe don't normalize
        if self._normalize_strategy is None:
            return features
        # Use the log zero guard for the sqrt zero guard
        guard_value = self._resolve_log_zero_guard_value(features.dtype)
        if self._normalize_strategy == "per_feature" or self._normalize_strategy == "all_features":
            # 'all_features' reduces over each sample; 'per_feature' reduces over each channel
            reduce_dim = 2
            if self._normalize_strategy == "all_features":
                reduce_dim = [1, 2]
            # [B, D, T] -> [B, D, 1] or [B, 1, 1]
            means = features.sum(dim=reduce_dim, keepdim=True).div(lengths.view(-1, 1, 1))
            stds = (
                features.sub(means)
                .masked_fill(mask, 0.0)
                .pow(2.0)
                .sum(dim=reduce_dim, keepdim=True)  # [B, D, T] -> [B, D, 1] or [B, 1, 1]
                .div(lengths.view(-1, 1, 1) - 1)  # assume biased estimator
                .clamp(min=guard_value)  # avoid sqrt(0)
                .sqrt()
            )
            features = (features - means) / (stds + eps)
        else:
            # Deprecating constant std/mean
            raise ValueError(f"Unsupported norm type: '{self._normalize_strategy}")
        features = features.masked_fill(mask, 0.0)
        return features

    def forward(self, input_signal: torch.Tensor, length: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        feature_lengths = self._compute_output_lengths(input_lengths=length)
        signals = self._apply_dithering(signals=input_signal)
        signals = self._apply_preemphasis(signals=signals)
        features = self._extract_spectrograms(signals=signals)
        features = self._apply_log(features=features)
        features = self._apply_normalization(features=features, lengths=feature_lengths)
        features = self._apply_pad_to(features=features)
        return features, feature_lengths  

def create_pre_processor(
    sample_rate=16000,
    window_size=0.025,
    window_stride=0.01,
    n_window_size=None,
    n_window_stride=None,
    window="hann",
    normalize="per_feature",
    n_fft=512,
    preemph=0.97,
    features=80,
    lowfreq=0,
    highfreq=None,
    log=True,
    log_zero_guard_type="add",
    log_zero_guard_value=2 ** -24,
    dither=0.00001,
    pad_to=0,
    frame_splicing=1,
    exact_pad=False,
    pad_value=0.0,
    mag_power=2.0,
    rng=None,
    nb_augmentation_prob=0.0,
    nb_max_freq=4000,
    use_torchaudio: bool = False,
    mel_norm="slaney",
    stft_exact_pad=False,
    stft_conv=False,
):
    _sample_rate = sample_rate
    if window_size and n_window_size:
        raise ValueError("Received both window_size and n_window_size. Only one should be specified.")
    if window_stride and n_window_stride:
        raise ValueError("Received both window_stride and n_window_stride. Only one should be specified.")
    if window_size:
        n_window_size = int(window_size * _sample_rate)
    if window_stride:
        n_window_stride = int(window_stride * _sample_rate)
    return FilterbankFeaturesTA(
        sample_rate=sample_rate,
        n_window_size=n_window_size,
        n_window_stride=n_window_stride,
        window=window,
        normalize=normalize,
        n_fft=n_fft,
        preemph=preemph,
        nfilt=features,
        lowfreq=lowfreq,
        highfreq=highfreq,
        log=log,
        log_zero_guard_type=log_zero_guard_type,
        log_zero_guard_value=log_zero_guard_value,
        dither=dither,
        pad_to=pad_to,
        frame_splicing=frame_splicing,
        exact_pad=exact_pad,
        pad_value=pad_value,
        mag_power=mag_power,
        rng=rng,
        nb_augmentation_prob=nb_augmentation_prob,
        nb_max_freq=nb_max_freq,
        mel_norm=mel_norm,
        stft_exact_pad=stft_exact_pad,
        stft_conv=stft_conv,
    )

def to_numpy(tensor):
    if tensor.requires_grad:
        return tensor.detach().stt_en_cconformer_ctc_large().numpy().astype(np.float32)
    else:
        return tensor.stt_en_cconformer_ctc_large().numpy().astype(np.float32)

# Load and resample the audio file using torchaudio
audio_file_path = "/data/asr/testing_audios/english/valid_and_test_data/ms400hr/audios/a007236be20b47dc9679e089891ee931.wav"
waveform, sample_rate = torchaudio.load(audio_file_path)
if sample_rate != SAMPLE_RATE:
    waveform = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=SAMPLE_RATE)(waveform)

# Initialize the pre-processing module
pre_processing_module = create_pre_processor()

# Initialize ONNX Runtime session
onnx_model_path = "/data/asr/ams/Conformer-CTC-BPE-v1_95_400-averaged.onnx"
ort_session = onnxruntime.InferenceSession(onnx_model_path)

# Initialize buffer
buffer = np.zeros((1, BUFFER_LEN))

# Initialize vocabulary and decoder
vocabulary = ['<unk>', 'e', '▁', 's', 'a', 't', 'i', 'd', '▁a', 'n', '▁the', 'l', 'y', 'u', 'o', 'm', 'p', '▁to', '▁s', 'h', '▁p', 'r', 'er', 'k', 'c', 're', '▁m', 'f', 'g', '▁in', '▁i', '▁of', 'ing', 'ar', '▁f', '▁w', '▁b', 'w', 'an', 'ed', 'in', '▁t', 'or', '▁and', '▁d', '▁on', 'b', '▁c', 'en', 'le', 've', 'ch', 'st', '▁he', '▁is', 'll', '▁be', '▁you', 'al', 'on', 'ro', '▁for', 'es', '▁co', '▁it', 'ur', 'at', '▁e', '▁g', '▁re', '▁ha', 'th', 'us', 'ra', '▁we', "▁'", '▁so', 'ent', 'ri', 'ce', '▁that', '▁at', 'it', 'ir', '▁o', '▁ma', '▁th', 'ic', '▁sh', 'ver', '▁do', 'j', '▁mo', 'ly', '▁st', '▁was', '▁ho', 'tion', 'ng', 'v', 'ow', 'ight', 'ter', 'x', '▁lo', 'vi', '▁not', '▁go', '▁with', '▁have', 'ck', '▁can', '▁what', '▁want', '▁no', '▁this', '▁will', '▁li', '▁his', '▁but', '▁two', '▁from', 'z', '▁book', '▁table', '▁out', 'q', "'"]
vocabulary.append("__")
decoder = ctc_decoder(
    lexicon="/data/asr/lms/12.0/merged_lm.lexicon",
    nbest=1,
    tokens=vocabulary,
    blank_token=vocabulary[-1],
    sil_token=vocabulary[-1],
    beam_size_token=50,
    lm="/data/asr/lms/12.0/merged_lm.bin",
    beam_threshold=20,
    beam_size=50,
    lm_weight=2,
    word_score=0
)

# Process audio in chunks
chunk_reader = AudioChunkIterator(waveform, CHUNK_LEN_SECS, SAMPLE_RATE)
buffer_list = []

for i, chunk in enumerate(chunk_reader):
    print(f"Processing chunk {i + 1}")
    # Shift buffer left and append new chunk
    buffer[:, :-CHUNK_LEN] = buffer[:, CHUNK_LEN:]
    buffer[:, -CHUNK_LEN:] = chunk

    # Convert buffer to tensor
    buffer_tensor = torch.tensor(buffer, dtype=torch.float32)

    # Process signal
    c_processed_signal, c_processed_signal_len = pre_processing_module.forward(buffer_tensor, length=torch.tensor([BUFFER_LEN]))
    c_processed_signal_numpy = to_numpy(c_processed_signal)
    c_processed_signal_len_numpy = c_processed_signal_len.stt_en_cconformer_ctc_large().numpy().astype(np.int64)

    # Prepare inputs for ONNX Runtime
    ort_inputs = {
        'audio_signal': c_processed_signal_numpy,
        'length': c_processed_signal_len_numpy
    }

    # Run inference with ONNX Runtime
    ologits = ort_session.run(None, ort_inputs)
    alogits = np.asarray(ologits)
    logits = torch.from_numpy(alogits[0])

    # Perform beam search decoding
    beam_search_result = decoder(logits)
    beam_search_transcript = " ".join(beam_search_result[0][0].words).strip()
    print(f"Transcript for chunk {i + 1}: {beam_search_transcript}")
    buffer_list.append(np.array(buffer))



Processing chunk 1
Transcript for chunk 1: 
Processing chunk 2
Transcript for chunk 2: 
Processing chunk 3
Transcript for chunk 3: 
Processing chunk 4
Transcript for chunk 4: 
Processing chunk 5
Transcript for chunk 5: 
Processing chunk 6
Transcript for chunk 6: 
Processing chunk 7
Transcript for chunk 7: 
Processing chunk 8
Transcript for chunk 8: 
Processing chunk 9
Transcript for chunk 9: india
Processing chunk 10
Transcript for chunk 10: india
Processing chunk 11
Transcript for chunk 11: india versus
Processing chunk 12
Transcript for chunk 12: india versus pakistan
Processing chunk 13
Transcript for chunk 13: india versus pakistan
Processing chunk 14
Transcript for chunk 14: india versus pakistan
Processing chunk 15
Transcript for chunk 15: india versus pakistan
Processing chunk 16
Transcript for chunk 16: india versus pakistan
Processing chunk 17
Transcript for chunk 17: india versus pakistan
Processing chunk 18
Transcript for chunk 18: india versus pakistan one
Processing chunk 

In [ ]:
import os
import onnxruntime
import torch
import numpy as np
import gc
import torchaudio
from torchaudio.models.decoder import ctc_decoder
import math
import random
from typing import Optional, Tuple, Union
import librosa
import numpy as np
import torch
import torch.nn as nn
import torchaudio

#CONSTANT = 1e-5


@torch.jit.script_if_tracing
def make_seq_mask_like(
    lengths: torch.Tensor, like: torch.Tensor, time_dim: int = -1, valid_ones: bool = True
) -> torch.Tensor:
    # Mask with shape [B, T]
    mask = torch.arange(like.shape[time_dim], device=like.device).repeat(lengths.shape[0], 1).lt(lengths.view(-1, 1))
    # [B, T] -> [B, *, T] where * is any number of singleton dimensions to expand to like tensor
    for _ in range(like.dim() - mask.dim()):
        mask = mask.unsqueeze(1)
    # If needed, transpose time dim
    if time_dim != -1 and time_dim != mask.dim() - 1:
        mask = mask.transpose(-1, time_dim)
    # Maybe invert the padded vs. valid token values
    if not valid_ones:
        mask = ~mask
    return mask


class FilterbankFeaturesTA(nn.Module):

    def __init__(
        self,
        sample_rate: int = 16000,
        n_window_size: int = 320,
        n_window_stride: int = 160,
        normalize: Optional[str] = "per_feature",
        nfilt: int = 64,
        n_fft: Optional[int] = None,
        preemph: float = 0.97,
        lowfreq: float = 0,
        highfreq: Optional[float] = None,
        log: bool = True,
        log_zero_guard_type: str = "add",
        log_zero_guard_value: Union[float, str] = 2 ** -24,
        dither: float = 1e-5,
        window: str = "hann",
        pad_to: int = 0,
        pad_value: float = 0.0,
        mel_norm="slaney",
        # Seems like no one uses these options anymore. Don't convolute the code by supporting thm.
        use_grads: bool = False,  # Deprecated arguments; kept for config compatibility
        max_duration: float = 16.7,  # Deprecated arguments; kept for config compatibility
        frame_splicing: int = 1,  # Deprecated arguments; kept for config compatibility
        exact_pad: bool = False,  # Deprecated arguments; kept for config compatibility
        nb_augmentation_prob: float = 0.0,  # Deprecated arguments; kept for config compatibility
        nb_max_freq: int = 4000,  # Deprecated arguments; kept for config compatibility
        mag_power: float = 2.0,  # Deprecated arguments; kept for config compatibility
        rng: Optional[random.Random] = None,  # Deprecated arguments; kept for config compatibility
        stft_exact_pad: bool = False,  # Deprecated arguments; kept for config compatibility
        stft_conv: bool = False,  # Deprecated arguments; kept for config compatibility
    ):
        super().__init__()

        # Make sure log zero guard is supported, if given as a string
        supported_log_zero_guard_strings = {"eps", "tiny"}
        if isinstance(log_zero_guard_value, str) and log_zero_guard_value not in supported_log_zero_guard_strings:
            raise ValueError(
                f"Log zero guard value must either be a float or a member of {supported_log_zero_guard_strings}"
            )

        # Copied from `AudioPreprocessor` due to the ad-hoc structuring of the Mel Spec extractor class
        self.torch_windows = {
            'hann': torch.hann_window
        }

        # Ensure we can look up the window function
        if window not in self.torch_windows:
            raise ValueError(f"Got window value '{window}' but expected a member of {self.torch_windows.keys()}")

        self.win_length = n_window_size
        self.hop_length = n_window_stride
        self._sample_rate = sample_rate
        self._normalize_strategy = normalize
        self._use_log = log
        self._preemphasis_value = preemph
        self.log_zero_guard_type = log_zero_guard_type
        self.log_zero_guard_value: Union[str, float] = log_zero_guard_value
        self.dither = dither
        self.pad_to = pad_to
        self.pad_value = pad_value
        self.n_fft = n_fft
        self._mel_spec_extractor: torchaudio.transforms.MelSpectrogram = torchaudio.transforms.MelSpectrogram(
            sample_rate=self._sample_rate,
            win_length=self.win_length,
            hop_length=self.hop_length,
            n_mels=nfilt,
            window_fn=self.torch_windows[window],
            mel_scale="slaney",
            norm=mel_norm,
            n_fft=n_fft,
            f_max=highfreq,
            f_min=lowfreq,
            wkwargs={"periodic": False},
        )

    @property
    def filter_banks(self):
        """ Matches the analogous class """
        return self._mel_spec_extractor.mel_scale.fb

    def _resolve_log_zero_guard_value(self, dtype: torch.dtype) -> float:
        if isinstance(self.log_zero_guard_value, float):
            return self.log_zero_guard_value
        return getattr(torch.finfo(dtype), self.log_zero_guard_value)

    def _apply_dithering(self, signals: torch.Tensor) -> torch.Tensor:
        if self.training and self.dither > 0.0:
            noise = torch.randn_like(signals) * self.dither
            signals = signals + noise
        return signals

    def _apply_preemphasis(self, signals: torch.Tensor) -> torch.Tensor:
        if self._preemphasis_value is not None:
            padded = torch.nn.functional.pad(signals, (1, 0))
            signals = signals - self._preemphasis_value * padded[:, :-1]
        return signals

    def _compute_output_lengths(self, input_lengths: torch.Tensor) -> torch.Tensor:
        out_lengths = input_lengths.div(self.hop_length, rounding_mode="floor").add(1).long()
        return out_lengths

    def _apply_pad_to(self, features: torch.Tensor) -> torch.Tensor:
        # Only apply during training; else need to capture dynamic shape for exported models
        if not self.training or self.pad_to == 0 or features.shape[-1] % self.pad_to == 0:
            return features
        pad_length = self.pad_to - (features.shape[-1] % self.pad_to)
        return torch.nn.functional.pad(features, pad=(0, pad_length), value=self.pad_value)

    def _apply_log(self, features: torch.Tensor) -> torch.Tensor:
        if self._use_log:
            zero_guard = self._resolve_log_zero_guard_value(features.dtype)
            if self.log_zero_guard_type == "add":
                features = features + zero_guard
            elif self.log_zero_guard_type == "clamp":
                features = features.clamp(min=zero_guard)
            else:
                raise ValueError(f"Unsupported log zero guard type: '{self.log_zero_guard_type}'")
            features = features.log()
        return features

    def _extract_spectrograms(self, signals: torch.Tensor) -> torch.Tensor:
        # Complex FFT needs to be done in single precision
        with torch.cuda.amp.autocast(enabled=False):
            features = self._mel_spec_extractor(waveform=signals)
        return features

    def _apply_normalization(self, features: torch.Tensor, lengths: torch.Tensor, eps: float = 1e-5) -> torch.Tensor:
        # For consistency, this function always does a masked fill even if not normalizing.
        mask: torch.Tensor = make_seq_mask_like(lengths=lengths, like=features, time_dim=-1, valid_ones=False)
        features = features.masked_fill(mask, 0.0)
        # Maybe don't normalize
        if self._normalize_strategy is None:
            return features
        # Use the log zero guard for the sqrt zero guard
        guard_value = self._resolve_log_zero_guard_value(features.dtype)
        if self._normalize_strategy == "per_feature" or self._normalize_strategy == "all_features":
            # 'all_features' reduces over each sample; 'per_feature' reduces over each channel
            reduce_dim = 2
            if self._normalize_strategy == "all_features":
                reduce_dim = [1, 2]
            # [B, D, T] -> [B, D, 1] or [B, 1, 1]
            means = features.sum(dim=reduce_dim, keepdim=True).div(lengths.view(-1, 1, 1))
            stds = (
                features.sub(means)
                .masked_fill(mask, 0.0)
                .pow(2.0)
                .sum(dim=reduce_dim, keepdim=True)  # [B, D, T] -> [B, D, 1] or [B, 1, 1]
                .div(lengths.view(-1, 1, 1) - 1)  # assume biased estimator
                .clamp(min=guard_value)  # avoid sqrt(0)
                .sqrt()
            )
            features = (features - means) / (stds + eps)
        else:
            # Deprecating constant std/mean
            raise ValueError(f"Unsupported norm type: '{self._normalize_strategy}")
        features = features.masked_fill(mask, 0.0)
        return features

    def forward(self, input_signal: torch.Tensor, length: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        feature_lengths = self._compute_output_lengths(input_lengths=length)
        signals = self._apply_dithering(signals=input_signal)
        signals = self._apply_preemphasis(signals=signals)
        features = self._extract_spectrograms(signals=signals)
        features = self._apply_log(features=features)
        features = self._apply_normalization(features=features, lengths=feature_lengths)
        features = self._apply_pad_to(features=features)
        return features, feature_lengths  

def create_pre_processor(
        sample_rate=16000,
        window_size=0.025,
        window_stride=0.01,
        n_window_size=None,
        n_window_stride=None,
        window="hann",
        normalize="per_feature",
        n_fft=512,
        preemph=0.97,
        features=80,
        lowfreq=0,
        highfreq=None,
        log=True,
        log_zero_guard_type="add",
        log_zero_guard_value=2 ** -24,
        dither=0.00001,
        pad_to=0,
        frame_splicing=1,
        exact_pad=False,
        pad_value=0.0,
        mag_power=2.0,
        rng=None,
        nb_augmentation_prob=0.0,
        nb_max_freq=4000,
        use_torchaudio: bool = False,
        mel_norm="slaney",
        stft_exact_pad=False,  # Deprecated arguments; kept for config compatibility
        stft_conv=False,  # Deprecated arguments; kept for config compatibility
):
    
    _sample_rate = sample_rate
    if window_size and n_window_size:
        raise ValueError(f"{self} received both window_size and " f"n_window_size. Only one should be specified.")
    if window_stride and n_window_stride:
        raise ValueError(
            f"{self} received both window_stride and " f"n_window_stride. Only one should be specified."
        )
    if window_size:
        n_window_size = int(window_size * _sample_rate)
    if window_stride:
        n_window_stride = int(window_stride * _sample_rate)
    return FilterbankFeaturesTA(
            sample_rate=sample_rate,
            n_window_size=n_window_size,
            n_window_stride=n_window_stride,
            window=window,
            normalize=normalize,
            n_fft=n_fft,
            preemph=preemph,
            nfilt=features,
            lowfreq=lowfreq,
            highfreq=highfreq,
            log=log,
            log_zero_guard_type=log_zero_guard_type,
            log_zero_guard_value=log_zero_guard_value,
            dither=dither,
            pad_to=pad_to,
            frame_splicing=frame_splicing,
            exact_pad=exact_pad,
            pad_value=pad_value,
            mag_power=mag_power,
            rng=rng,
            nb_augmentation_prob=nb_augmentation_prob,
            nb_max_freq=nb_max_freq,
            mel_norm=mel_norm,
            stft_exact_pad=stft_exact_pad,  # Deprecated arguments; kept for config compatibility
            stft_conv=stft_conv,  # Deprecated arguments; kept for config compatibility
        )

# convert torch tensor to numpy array with dtype float32
def to_numpy_len(tensor):
    if tensor.requires_grad:
        return tensor.detach().stt_en_cconformer_ctc_large().numpy().astype(np.int64)
    else:
        return tensor.stt_en_cconformer_ctc_large().numpy().astype(np.int64)

def to_numpy_signal(tensor):
    if tensor.requires_grad:
        return tensor.detach().stt_en_cconformer_ctc_large().numpy().astype(np.float32)
    else:
        return tensor.stt_en_cconformer_ctc_large().numpy().astype(np.float32)
    

vocabulary = ['<unk>', 'e', '▁', 's', 'a', 't', 'i', 'd', '▁a', 'n', '▁the', 'l', 'y', 'u', 'o', 'm', 'p', '▁to', '▁s', 'h', '▁p', 'r', 'er', 'k', 'c', 're', '▁m', 'f', 'g', '▁in', '▁i', '▁of', 'ing', 'ar', '▁f', '▁w', '▁b', 'w', 'an', 'ed', 'in', '▁t', 'or', '▁and', '▁d', '▁on', 'b', '▁c', 'en', 'le', 've', 'ch', 'st', '▁he', '▁is', 'll', '▁be', '▁you', 'al', 'on', 'ro', '▁for', 'es', '▁co', '▁it', 'ur', 'at', '▁e', '▁g', '▁re', '▁ha', 'th', 'us', 'ra', '▁we', "▁'", '▁so', 'ent', 'ri', 'ce', '▁that', '▁at', 'it', 'ir', '▁o', '▁ma', '▁th', 'ic', '▁sh', 'ver', '▁do', 'j', '▁mo', 'ly', '▁st', '▁was', '▁ho', 'tion', 'ng', 'v', 'ow', 'ight', 'ter', 'x', '▁lo', 'vi', '▁not', '▁go', '▁with', '▁have', 'ck', '▁can', '▁what', '▁want', '▁no', '▁this', '▁will', '▁li', '▁his', '▁but', '▁two', '▁from', 'z', '▁book', '▁table', '▁out', 'q', "'"]
vocabulary.append("__")
#print("vocab",vocabulary)

# Initialize ONNX Runtime session
onnx_model_path = "/data/asr/ams/Conformer-CTC-BPE-v1_95_400-averaged.onnx"
ort_session = onnxruntime.InferenceSession(onnx_model_path)

# Audio file path
audio_file_path = "/data/asr/testing_audios/english/valid_and_test_data/ms400hr/audios/a007236be20b47dc9679e089891ee931.wav"

# Load and resample the audio file using torchaudio
waveform, sample_rate = torchaudio.load(audio_file_path)
if sample_rate != 16000:
    waveform = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)(waveform)

pre_processing_module = create_pre_processor()

c_processed_signal, c_processed_signal_len = pre_processing_module.forward(waveform, length=torch.tensor([waveform.shape[1]]))


# Convert processed signal and its length to numpy arrays for ONNX Runtime
c_processed_signal_numpy = to_numpy_signal(c_processed_signal)
c_processed_signal_len_numpy = to_numpy_len(c_processed_signal_len)


# Prepare inputs for ONNX Runtime
ort_inputs = {
    'audio_signal': c_processed_signal_numpy,
    'length': c_processed_signal_len_numpy
}
# Run inference with ONNX Runtime
ologits = ort_session.run(None, ort_inputs)
alogits = np.asarray(ologits)
logits = torch.from_numpy(alogits[0])


decoder = ctc_decoder(lexicon="/data/asr/lms/12.0/merged_lm.lexicon",
    nbest= 1,
    tokens=vocabulary,
    blank_token=vocabulary[-1],
    sil_token=vocabulary[-1],
    beam_size_token= 50,
    lm="/data/asr/lms/12.0/merged_lm.bin",
    beam_threshold=20,
    beam_size = 50,
    lm_weight=2,       
    word_score=0)



beam_search_result = decoder(logits)
#print("beam",beam_search_result)
beam_search_transcript = " ".join(beam_search_result[0][0].words).strip()


print(f"Transcript: {beam_search_transcript}")

Transcript: india versus pakistan world cup final


With Lexicon and LM

In [ ]:
import os
import onnxruntime
import torch
import numpy as np
import gc
import torchaudio
import nemo.collections.asr as nemo_asr
from torchaudio.models.decoder import ctc_decoder

# convert torch tensor to numpy array with dtype float32
def to_numpy(tensor):
    if tensor.requires_grad:
        return tensor.detach().stt_en_cconformer_ctc_large().numpy().astype(np.int64)
    else:
        return tensor.stt_en_cconformer_ctc_large().numpy().astype(np.int64)

def to_numpy_signal(tensor):
    if tensor.requires_grad:
        return tensor.detach().stt_en_cconformer_ctc_large().numpy().astype(np.float32)
    else:
        return tensor.stt_en_cconformer_ctc_large().numpy().astype(np.float32)
    
torch.cuda.empty_cache()
gc.collect()

# Restore ASR model from checkpoint
model_path = "/data/asr/ams/Conformer-CTC-BPE-v1_95_400.nemo"
model = nemo_asr.models.EncDecCTCModelBPE.restore_from(model_path)
device = torch.device("stt_en_cconformer_ctc_large")
model = model.to(device)

vocabulary = model.decoder.vocabulary
vocabulary.append("__")
#print("vocab",vocabulary)

# Initialize ONNX Runtime session
onnx_model_path = "/data/asr/ams/Conformer-CTC-BPE-v1_95_400-averaged.onnx"
ort_session = onnxruntime.InferenceSession(onnx_model_path)

# Audio file path
audio_file_path = "/data/asr/adapter_training_data/english/bugs83/bugs83_audios/2a3c92955d10424a9272a072b90930e4.wav"

# Load and resample the audio file using torchaudio
waveform, sample_rate = torchaudio.load(audio_file_path)
if sample_rate != 16000:
    waveform = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)(waveform)


processed_signal, processed_signal_len = model.preprocessor(input_signal=waveform, length=torch.tensor([waveform.shape[1]]))

# Convert processed signal and its length to numpy arrays for ONNX Runtime
processed_signal_numpy = to_numpy_signal(processed_signal)
processed_signal_len_numpy = to_numpy(processed_signal_len)

# Prepare inputs for ONNX Runtime
ort_inputs = {
    'audio_signal': processed_signal_numpy,
    'length': processed_signal_len_numpy
}
# Run inference with ONNX Runtime
ologits = ort_session.run(None, ort_inputs)
alogits = np.asarray(ologits)
logits = torch.from_numpy(alogits[0])



decoder = ctc_decoder(lexicon="/data/asr/lms/12.0/merged_lm.lexicon",
    nbest= 3,
    tokens=vocabulary,
    blank_token=vocabulary[-1],
    sil_token=vocabulary[-1],
    beam_size_token= 50,
    lm="/data/asr/lms/12.0/merged_lm.bin",
    beam_threshold=20,
    beam_size = 50,
    lm_weight=1,       
    word_score=0)

# Decode logits
results = decoder(logits)
#print("results", results)

beam_search_result = decoder(logits)
print("beam",beam_search_result)
beam_search_transcript = " ".join(beam_search_result[0][0].words).strip()

#print(decoder.idxs_to_tokens)

#tokens_str = "".join(decoder.idxs_to_tokens(beam_search_result[0][0].tokens))
#print(tokens_str)
#transcript = " ".join(tokens_str.split("_"))
#print("final",transcript)
#transcripts = [" ".join(hypo[0].words) for hypo in results]
#print("Transcripts with lexicon:", transcripts)
print(f"Transcript: {beam_search_transcript}")
# Convert token indices to tokens
batch_tokens = [decoder.idxs_to_tokens(hypo[0].tokens) for hypo in results]
print("batch_token", batch_tokens)
transcripts_without_lexicon = ["".join(tokens) for tokens in batch_tokens]
print("Transcripts without lexicon:", transcripts_without_lexicon)



[NeMo W 2024-05-09 10:01:05 experimental:27] Module <class 'nemo.collections.asr.modules.audio_modules.SpectrogramToMultichannelFeatures'> is experimental, not ready for production and is not fully supported. Use at your own risk.


[NeMo I 2024-05-09 10:01:06 mixins:170] Tokenizer SentencePieceTokenizer initialized with 128 tokens


[NeMo W 2024-05-09 10:01:06 modelPT:161] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath:
    - /data/asr/training_data_95/trimmed_morethan95/training_ms400_ms4200.jsonl
    sample_rate: 16000
    batch_size: 64
    shuffle: true
    num_workers: 8
    pin_memory: true
    max_duration: 11
    min_duration: 0.5
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_batch_size: None
    
[NeMo W 2024-05-09 10:01:06 modelPT:168] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath:
    - /data/asr/training_data_95/trimmed_morethan95/validation_ms400_ms4200.jsonl
    sample_rate: 16000
    batch

[NeMo I 2024-05-09 10:01:06 features:291] PADDING: 0
[NeMo I 2024-05-09 10:01:08 save_restore_connector:249] Model EncDecCTCModelBPE was successfully restored from /data/asr/ams/Conformer-CTC-BPE-v1_95_400.nemo.
beam [[CTCHypothesis(tokens=tensor([85,  4,  2, 91, 14, 98]), words=['maa', 'jong'], score=-15.04334778293029, timesteps=tensor([31, 38, 43, 44, 50, 54], dtype=torch.int32)), CTCHypothesis(tokens=tensor([85, 40,  2, 91, 14, 98]), words=['main', 'jong'], score=-16.586206726273453, timesteps=tensor([31, 38, 43, 44, 50, 54], dtype=torch.int32)), CTCHypothesis(tokens=tensor([85, 40,  4,  2, 91, 14, 98]), words=['maina', 'jong'], score=-17.559996716221725, timesteps=tensor([31, 38, 39, 43, 44, 50, 54], dtype=torch.int32))]]
Transcript: maa jong
batch_token [['▁ma', 'a', '▁', 'j', 'o', 'ng']]
Transcripts without lexicon: ['▁maa▁jong']


Without Lexicon and LM (Greedy)

In [ ]:
import os
import onnxruntime
import torch
import numpy as np
import gc
import soundfile as sf
import nemo.collections.asr as nemo_asr
from nemo.collections.asr.metrics.wer import WER


def to_numpy(tensor):
    return tensor.detach().stt_en_cconformer_ctc_large().numpy() if tensor.requires_grad else tensor.stt_en_cconformer_ctc_large().numpy()

# Clear up memory
torch.cuda.empty_cache()
gc.collect()
model = nemo_asr.models.EncDecCTCModelBPE.restore_from("/data/asr/ams/Conformer-CTC-BPE-v1_95_400.nemo",map_location = 'cuda')

device = torch.device('cuda' if torch.cuda.is_available() else 'stt_en_cconformer_ctc_large')
model = model.to(device)


# Initialize ONNX Runtime session
ort_session = onnxruntime.InferenceSession("/data/asr/ams/Conformer-CTC-BPE-v1_95_400-averaged.onnx")

audio_file_path = "/data/asr/testing_audios/english/valid_and_test_data/ms400hr/audios/a007236be20b47dc9679e089891ee931.wav"
# Load the audio file
signal, sample_rate = sf.read(audio_file_path, dtype='float32')
if sample_rate != 16000:
    raise ValueError(f"Sample rate of the file is {sample_rate}, but 16000 Hz is expected.")

input_signal = torch.tensor(signal[None, :], dtype=torch.float32).to(device)

# Process the audio file using the  preprocessor
processed_signal, processed_signal_len = model.preprocessor(input_signal=input_signal, length=torch.tensor([input_signal.shape[1]], dtype=torch.long).to(device))

processed_signal_numpy = to_numpy(processed_signal)
processed_signal_len_numpy = to_numpy(processed_signal_len)

# ONNX Runtime Inference
ort_inputs = {
    'audio_signal': processed_signal_numpy,
    'length': processed_signal_len_numpy
}

ologits = ort_session.run(None, ort_inputs)
alogits = np.asarray(ologits)
logits = torch.from_numpy(alogits[0])



# Check vocabulary size and predictions
vocabulary = model.decoder.vocabulary


from pyctcdecode import build_ctcdecoder
decoder = build_ctcdecoder(vocabulary,kenlm_model_path="/data/asr/_users/vikesh/lm/output/merged_lm.bin",alpha=2.0,beta=0)

probs = torch.nn.functional.softmax(logits, dim=-1).numpy()

decoded_transcription_lm = decoder.decode( probs[0],beam_width=50)

print("Transcription:", decoded_transcription_lm)


[NeMo I 2024-04-30 10:03:23 mixins:170] Tokenizer SentencePieceTokenizer initialized with 128 tokens


[NeMo W 2024-04-30 10:03:23 modelPT:161] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath:
    - /data/asr/training_data_95/trimmed_morethan95/training_ms400_ms4200.jsonl
    sample_rate: 16000
    batch_size: 64
    shuffle: true
    num_workers: 8
    pin_memory: true
    max_duration: 11
    min_duration: 0.5
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_batch_size: None
    
[NeMo W 2024-04-30 10:03:23 modelPT:168] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath:
    - /data/asr/training_data_95/trimmed_morethan95/validation_ms400_ms4200.jsonl
    sample_rate: 16000
    batch

[NeMo I 2024-04-30 10:03:23 features:291] PADDING: 0
[NeMo I 2024-04-30 10:03:24 save_restore_connector:249] Model EncDecCTCModelBPE was successfully restored from /data/asr/ams/Conformer-CTC-BPE-v1_95_400.nemo.


W0430 10:03:26.560025 139691802486592 decoder.py:914] Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
W0430 10:03:26.564910 139691802486592 language_model.py:228] No known unigrams provided, decoding results might be a lot worse.


Transcription: india versus pakistan world cup final
